![Week 6 Day 4](../../assets/banners/w6d4.png)

# Week 6 Day 4 - Solution: A Render Blueprint, With Checks

**The exercise:** describe both GradeWise services in a `render.yaml` Blueprint (API with a disk and a health check, UI with the Streamlit command and `API_URL`), load it back, and write checks that would catch a broken deploy.

Try the exercise yourself first: struggling a little is where the learning happens! Then compare with this one. Everything runs offline in 🧪 demo mode.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ No OpenAI API Key set. That's fine this week: every cell runs offline in 🧪 demo mode.")
    print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

In [ ]:
import yaml

from gradewise.config import OUTPUT_DIR

OUT = OUTPUT_DIR / "week6" / "day4_solution"
OUT.mkdir(parents=True, exist_ok=True)

RENDER_YAML = """
# render.yaml: GradeWise on Render, as code. Commit it at the repo root, then New → Blueprint in Render.
services:
  - type: web
    name: gradewise-api
    runtime: docker
    dockerfilePath: ./docker/Dockerfile
    dockerContext: .
    plan: starter                      # disks need a paid instance
    healthCheckPath: /health
    envVars:
      - key: API_PORT
        value: "10000"                 # Render routes traffic to $PORT, 10000 by default
      - key: GRADEWISE_DEMO
        value: "true"                  # a public demo: no model spend
      - key: OPENAI_API_KEY
        sync: false                    # 🔑 a secret: you type it in Render's dashboard, never in git
    disk:
      name: gradewise-output
      mountPath: /app/output
      sizeGB: 1

  - type: web
    name: gradewise-ui
    runtime: docker
    dockerfilePath: ./docker/Dockerfile
    dockerContext: .
    plan: starter
    dockerCommand: python -m streamlit run gradewise/ui/app.py --server.address=0.0.0.0 --server.port=10000
    healthCheckPath: /_stcore/health
    envVars:
      - key: API_URL
        value: https://gradewise-api.onrender.com   # ✋ your API service's URL
"""
path = OUT / "render.yaml"
path.write_text(RENDER_YAML.lstrip(), encoding="utf-8")
blueprint = yaml.safe_load(path.read_text(encoding="utf-8"))
print(f"📝 {path} · services: {[s['name'] for s in blueprint['services']]}")

### Checks that would catch a broken deploy
Each check encodes one lesson from Day 4. Writing them as a function that returns a list of problems (like `ExamPaper.problems()` in week 1) means we can test it on a deliberately broken copy too.

In [ ]:
import copy
import re


def blueprint_problems(bp: dict) -> list[str]:
    problems = []
    services = {s["name"]: s for s in bp.get("services", [])}
    api, ui = services.get("gradewise-api"), services.get("gradewise-ui")
    if not api or not ui:
        return ["both gradewise-api and gradewise-ui must be defined"]
    env = {s["name"]: {e["key"]: e for e in s.get("envVars", [])} for s in (api, ui)}

    if api.get("disk", {}).get("mountPath") != "/app/output":
        problems.append("api: no disk at /app/output, so every redeploy wipes exams and reports")
    if env["gradewise-api"].get("API_PORT", {}).get("value") != "10000":
        problems.append("api: API_PORT must match Render's port (10000)")
    if api.get("healthCheckPath") != "/health" or ui.get("healthCheckPath") != "/_stcore/health":
        problems.append("health checks: api → /health, ui → /_stcore/health")
    if "--server.port=10000" not in ui.get("dockerCommand", "") or "0.0.0.0" not in ui.get("dockerCommand", ""):
        problems.append("ui: Streamlit must listen on 0.0.0.0:10000")
    if not env["gradewise-ui"].get("API_URL", {}).get("value", "").startswith("http"):
        problems.append("ui: API_URL must be the API's full URL")
    for service, variables in env.items():
        for key, var in variables.items():
            value = str(var.get("value", ""))
            if re.match(r"(sk-|sk_|AIza|ghp_)", value) or (key.endswith("_KEY") and "value" in var):
                problems.append(f"{service}: {key} has a committed secret value; use sync: false")
    return problems


print("our blueprint:", blueprint_problems(blueprint) or "✅ no problems")

broken = copy.deepcopy(blueprint)
del broken["services"][0]["disk"]
broken["services"][0]["envVars"][2] = {"key": "OPENAI_API_KEY", "value": "sk-proj-oops"}
broken["services"][1]["dockerCommand"] = "python -m streamlit run gradewise/ui/app.py"
print("\nbroken copy:")
for problem in blueprint_problems(broken):
    print("  ❌", problem)

💡 Run checks like these in CI on every pull request and a broken deploy never reaches `main`. The same idea works for `fly.toml` (`tomllib` is in the standard library) and for `docker-compose.yml` (Day 4's pre-flight checklist).

🚀 Render can also pass one service's address to another with `fromService` (for example its private `hostport`). Check Render's Blueprint reference for the exact keys before relying on them, and remember `gradewise/ui/client.py` wants a full URL with `http://` or `https://`.